# TalentCLEF 2026 Task A - PyTerrier Experiments

This notebook is the main experiment table for English and Spanish Task A runs in `this repository`.

It now keeps query augmentation outside the notebook in `task_a_query_augmentation.py`.
That gives us:
- deterministic augmentation rules
- cached JSON outputs per query under `var/cache/query_augmentation/`
- reusable query views for both exploration and final submission runs

Current experiment families:
- BM25 over a PyTerrier text index
- JobBERT dense retrieval baseline
- deterministic JobBERT query augmentation views
- optional SPLADE retrieval if `pyt_splade` is installed
- optional JobBERT + SPLADE fusion / reranking


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "task_a_jobbert_dense_en_es.py").exists():
    if (PROJECT_ROOT.parent / "task_a_jobbert_dense_en_es.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the repository root or the notebooks/ directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from task_a_jobbert_dense_en_es import (
    MODEL_NAME,
    build_dense_run,
    evaluate_run,
    load_task_a_inputs,
    load_task_a_split,
    sanitize_token,
    save_trec_run,
    zip_submission,
)
from task_a_query_augmentation import (
    AUGMENTATION_VERSION,
    augment_topics,
    build_query_view,
    summarize_augmentation,
)
from task_a_colbert_rerank import (
    COLBERT_MODEL_NAME_DEFAULT,
    build_colbert_chunked_rerank_run,
    build_colbert_rerank_run,
    splice_reranked_candidates,
)

PROJECT_ROOT


In [ ]:
import pandas as pd

try:
    import pyterrier as pt
except Exception as exc:
    raise RuntimeError(
        "PyTerrier is not installed. Install it first, for example with `pip install pyterrier`."
    ) from exc

try:
    import pyt_splade
    HAS_SPLADE = True
except Exception:
    pyt_splade = None
    HAS_SPLADE = False

if hasattr(pt, "started") and not pt.started():
    try:
        pt.java.init()
    except Exception:
        pt.init()

print("PyTerrier started:", pt.started() if hasattr(pt, "started") else "unknown")
print("SPLADE available:", HAS_SPLADE)
print("Augmentation version:", AUGMENTATION_VERSION)


In [ ]:
ART_ROOT = PROJECT_ROOT / "var" / "artifacts" / "pyterrier_experiments"
SUBMISSION_ROOT = PROJECT_ROOT / "var" / "artifacts" / "pyterrier_submission"
AUGMENTATION_CACHE_ROOT = PROJECT_ROOT / "var" / "cache" / "query_augmentation"
AUGMENTATION_SUMMARY_ROOT = PROJECT_ROOT / "var" / "artifacts" / "query_augmentation"

LANG_CONFIGS = {
    "en": {
        "development_dir": PROJECT_ROOT / "data" / "development" / "en",
        "test_dir": PROJECT_ROOT / "data" / "test" / "en",
        "submission_lang_pair": "en-en",
    },
    "es": {
        "development_dir": PROJECT_ROOT / "data" / "development" / "es",
        "test_dir": PROJECT_ROOT / "data" / "test" / "es",
        "submission_lang_pair": "es-es",
    },
}

TEAM_NAME = "yourteam"  # change before official submission
APPROACH_ID = "pt_bm25"
JOBBERT_BATCH_SIZE_DOC = 16
JOBBERT_BATCH_SIZE_QUERY = 16
K_CAND = 100
K_FINAL = 50
FUSION_RRF_K = 60
RERANK_JOBBERT_WEIGHT = 2.0
RERANK_SPLADE_WEIGHT = 1.0
COMPACT_RRF_JOBBERT_WEIGHT = 2.0
COMPACT_RRF_REWRITE_WEIGHT = 1.0
USE_SPLADE = HAS_SPLADE
USE_COLBERT = False
USE_COLBERT_CHUNKED = False
COLBERT_MODEL_NAME = COLBERT_MODEL_NAME_DEFAULT
COLBERT_DEVICE = "cpu"
COLBERT_CAND_K = 50
COLBERT_QUERY_VIEW = "ORIGINAL"
COLBERT_QUERY_LENGTH = 128
COLBERT_DOCUMENT_LENGTH = 220
COLBERT_QUERY_BATCH_SIZE = 4
COLBERT_DOCUMENT_BATCH_SIZE = 2
COLBERT_CHUNK_SIZE_WORDS = 160
COLBERT_CHUNK_STRIDE_WORDS = 120
COLBERT_MAX_CHUNKS_PER_DOC = 5
COLBERT_EXPERIMENTS = [
    ("JobBERT>>ColBERT", "ORIGINAL"),
    ("JobBERT+CompactRewrite>>ColBERT", "COMPACT_REWRITE"),
]
AUGMENTATION_OVERWRITE = False
JOBBERT_AUGMENTED_VIEWS = [
    ("JobBERT+CompactRewrite", "COMPACT_REWRITE"),
    ("JobBERT+Profile", "PROFILE"),
    ("JobBERT+IdealResume", "IDEAL_RESUME"),
    ("JobBERT+Profile+IdealResume", "PROFILE_IDEAL_RESUME"),
    ("JobBERT+Profile+IdealResume+Skills", "PROFILE_IDEAL_RESUME_SKILLS"),
]
RUN_TEST_SUBMISSION = False
TEST_PIPELINE = "BM25"  # or "JOBBERT" / "JOBBERT_COLBERT_RERANK" / "JOBBERT_COLBERT_COMPACT_RERANK" / "JOBBERT_COLBERT_CHUNKED_RERANK" / "JOBBERT_COMPACT_REWRITE" / "JOBBERT_COMPACT_REWRITE_RRF" / "JOBBERT_PROFILE" / "JOBBERT_IDEAL_RESUME" / "JOBBERT_PROFILE_IDEAL_RESUME" / "JOBBERT_PROFILE_IDEAL_RESUME_SKILLS" / "SPLADE" / "JOBBERT_SPLADE_RRF" / "JOBBERT_SPLADE_RERANK"


def build_pt_corpus_iter(corpus_df: pd.DataFrame):
    for _, row in corpus_df.iterrows():
        yield {"docno": str(row["docno"]), "text": str(row["text"])}


def build_or_load_pt_index(corpus_df: pd.DataFrame, index_dir: Path):
    data_properties = index_dir / "data.properties"
    if data_properties.exists():
        return pt.IndexFactory.of(str(index_dir))

    indexer = pt.IterDictIndexer(
        str(index_dir),
        meta={"docno": 64, "text": 4096},
        text_attrs=["text"],
        overwrite=True,
        stemmer="none",
        stopwords=None,
        tokeniser="english",
    )
    return indexer.index(build_pt_corpus_iter(corpus_df))


def build_or_load_splade_index(corpus_df: pd.DataFrame, index_dir: Path):
    if not HAS_SPLADE:
        raise RuntimeError("pyt_splade is not installed.")

    data_properties = index_dir / "data.properties"
    if data_properties.exists():
        return pt.IndexFactory.of(str(index_dir))

    splade = pyt_splade.Splade()
    try:
        splade_indexer = pt.IterDictIndexer(
            str(index_dir),
            pretokenised=True,
            meta={"docno": 64},
            overwrite=True,
            stemmer="none",
            stopwords=None,
            tokeniser="identity",
        )
    except Exception:
        splade_indexer = pt.IterDictIndexer(
            str(index_dir),
            pretokenised=True,
            meta={"docno": 64},
            overwrite=True,
            stemmer="none",
            stopwords=None,
        )

    index_pipe = splade.doc_encoder() >> splade_indexer
    index_ref = index_pipe.index(build_pt_corpus_iter(corpus_df), batch_size=32)
    return index_ref


def pt_run_to_standard_run(df: pd.DataFrame, k: int) -> pd.DataFrame:
    out = df.copy()
    out["qid"] = out["qid"].astype(str)
    out["docno"] = out["docno"].astype(str)
    out["score"] = out["score"].astype(float)
    out = out.sort_values(["qid", "score"], ascending=[True, False]).reset_index(drop=True)
    out["rank"] = out.groupby("qid").cumcount() + 1
    return out.groupby("qid", sort=False).head(k)[["qid", "docno", "score", "rank"]].copy()


def cut_run_to_k(df: pd.DataFrame, k: int) -> pd.DataFrame:
    out = df.copy()
    out["qid"] = out["qid"].astype(str)
    out["docno"] = out["docno"].astype(str)
    out = out.sort_values(["qid", "rank"], ascending=[True, True]).reset_index(drop=True)
    return out.groupby("qid", sort=False).head(k)[["qid", "docno", "score", "rank"]].copy()


def rrf_fuse_runs(named_runs, final_k: int, rrf_k: int = 60, weights=None) -> pd.DataFrame:
    weights = weights or {}
    qids = sorted({str(qid) for _, df in named_runs for qid in df["qid"].astype(str).tolist()})
    rows = []

    for qid in qids:
        fused_scores = {}
        for name, df in named_runs:
            weight = float(weights.get(name, 1.0))
            sub = df[df["qid"].astype(str) == str(qid)].sort_values("rank")
            for _, row in sub.iterrows():
                docno = str(row["docno"])
                rank = int(row["rank"])
                fused_scores[docno] = fused_scores.get(docno, 0.0) + (weight / (rrf_k + rank))

        ranked = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)[:final_k]
        for rank, (docno, score) in enumerate(ranked, start=1):
            rows.append({"qid": str(qid), "docno": docno, "score": float(score), "rank": int(rank)})

    return pd.DataFrame(rows, columns=["qid", "docno", "score", "rank"])


def rerank_candidates_with_secondary_rrf(
    primary_candidates: pd.DataFrame,
    secondary_full_run: pd.DataFrame,
    final_k: int,
    rrf_k: int = 60,
    primary_weight: float = 2.0,
    secondary_weight: float = 1.0,
) -> pd.DataFrame:
    secondary_rank = {
        (str(row.qid), str(row.docno)): int(row.rank)
        for row in secondary_full_run[["qid", "docno", "rank"]].itertuples(index=False)
    }

    rows = []
    for qid, group in primary_candidates.groupby("qid", sort=False):
        scored = []
        group = group.sort_values("rank")
        for row in group.itertuples(index=False):
            docno = str(row.docno)
            p_rank = int(row.rank)
            s_rank = secondary_rank.get((str(qid), docno))
            score = primary_weight / (rrf_k + p_rank)
            if s_rank is not None:
                score += secondary_weight / (rrf_k + s_rank)
            scored.append((docno, score))

        scored = sorted(scored, key=lambda item: item[1], reverse=True)[:final_k]
        for rank, (docno, score) in enumerate(scored, start=1):
            rows.append({"qid": str(qid), "docno": docno, "score": float(score), "rank": int(rank)})

    return pd.DataFrame(rows, columns=["qid", "docno", "score", "rank"])


def build_jobbert_run_for_topics(corpus: pd.DataFrame, topics_df: pd.DataFrame, k: int) -> pd.DataFrame:
    return build_dense_run(
        corpus=corpus,
        topics=topics_df,
        model_name=MODEL_NAME,
        batch_size_doc=JOBBERT_BATCH_SIZE_DOC,
        batch_size_query=JOBBERT_BATCH_SIZE_QUERY,
        k=k,
    )


def ensure_augmented_topics(topics: pd.DataFrame, lang: str, split_name: str) -> pd.DataFrame:
    topics_aug = augment_topics(
        topics=topics,
        lang=lang,
        split_name=split_name,
        cache_root=AUGMENTATION_CACHE_ROOT,
        overwrite=AUGMENTATION_OVERWRITE,
    )
    summary_path = AUGMENTATION_SUMMARY_ROOT / split_name / f"{lang}_query_augmentation_summary.csv"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summarize_augmentation(topics_aug).to_csv(summary_path, index=False)
    return topics_aug


def run_dev_experiments(lang: str, cfg: dict) -> pd.DataFrame:
    corpus, topics, qrels = load_task_a_split(cfg["development_dir"])
    topics_aug = ensure_augmented_topics(topics, lang, "development")
    art_dir = ART_ROOT / lang
    art_dir.mkdir(parents=True, exist_ok=True)

    pt_index_dir = art_dir / "pt_text_index"
    pt_index = build_or_load_pt_index(corpus, pt_index_dir)
    topics_pt = build_query_view(topics_aug, "ORIGINAL")

    runs = []

    bm25 = pt.terrier.Retriever(pt_index, wmodel="BM25")
    bm25_run = pt_run_to_standard_run(bm25.transform(topics_pt), K_FINAL)
    runs.append(("BM25", bm25_run))

    jobbert_cand_run = build_jobbert_run_for_topics(corpus, topics_pt, K_CAND)
    jobbert_run = cut_run_to_k(jobbert_cand_run, K_FINAL)
    runs.append(("JobBERT", jobbert_run))

    if USE_COLBERT:
        colbert_candidates = cut_run_to_k(jobbert_cand_run, COLBERT_CAND_K)
        for pipeline_name, query_view in COLBERT_EXPERIMENTS:
            colbert_run = build_colbert_rerank_run(
                corpus=corpus,
                topics=build_query_view(topics_aug, query_view),
                primary_candidates=colbert_candidates,
                final_k=K_FINAL,
                model_name=COLBERT_MODEL_NAME,
                device=COLBERT_DEVICE,
                query_length=COLBERT_QUERY_LENGTH,
                document_length=COLBERT_DOCUMENT_LENGTH,
                query_batch_size=COLBERT_QUERY_BATCH_SIZE,
                document_batch_size=COLBERT_DOCUMENT_BATCH_SIZE,
            )
            runs.append((pipeline_name, colbert_run))

    if USE_COLBERT_CHUNKED:
        colbert_candidates = cut_run_to_k(jobbert_cand_run, COLBERT_CAND_K)
        chunked_colbert_run = build_colbert_chunked_rerank_run(
            corpus=corpus,
            topics=build_query_view(topics_aug, COLBERT_QUERY_VIEW),
            primary_candidates=colbert_candidates,
            final_k=K_FINAL,
            model_name=COLBERT_MODEL_NAME,
            device=COLBERT_DEVICE,
            query_length=COLBERT_QUERY_LENGTH,
            document_length=COLBERT_DOCUMENT_LENGTH,
            query_batch_size=COLBERT_QUERY_BATCH_SIZE,
            document_batch_size=COLBERT_DOCUMENT_BATCH_SIZE,
            chunk_size_words=COLBERT_CHUNK_SIZE_WORDS,
            chunk_stride_words=COLBERT_CHUNK_STRIDE_WORDS,
            max_chunks_per_doc=COLBERT_MAX_CHUNKS_PER_DOC,
        )
        runs.append(("JobBERT>>ColBERTChunked", chunked_colbert_run))

    jobbert_augmented_runs = {}
    for pipeline_name, view_name in JOBBERT_AUGMENTED_VIEWS:
        view_topics = build_query_view(topics_aug, view_name)
        aug_cand_run = build_jobbert_run_for_topics(corpus, view_topics, K_CAND)
        aug_run = cut_run_to_k(aug_cand_run, K_FINAL)
        jobbert_augmented_runs[view_name] = aug_run
        runs.append((pipeline_name, aug_run))

    compact_rrf_run = rrf_fuse_runs(
        [("JobBERT", jobbert_run), ("CompactRewrite", jobbert_augmented_runs["COMPACT_REWRITE"])],
        final_k=K_FINAL,
        rrf_k=FUSION_RRF_K,
        weights={"JobBERT": COMPACT_RRF_JOBBERT_WEIGHT, "CompactRewrite": COMPACT_RRF_REWRITE_WEIGHT},
    )
    runs.append(("JobBERT+CompactRewrite_RRF", compact_rrf_run))

    if USE_SPLADE and HAS_SPLADE:
        splade_index_dir = art_dir / "pt_splade_index"
        splade_index = build_or_load_splade_index(corpus, splade_index_dir)
        splade = pyt_splade.Splade()
        splade_retriever = splade.query_encoder() >> pt.terrier.Retriever(splade_index, wmodel="Tf")
        splade_full_run = pt_run_to_standard_run(splade_retriever.transform(topics_pt), len(corpus))
        splade_run = cut_run_to_k(splade_full_run, K_FINAL)
        runs.append(("SPLADE", splade_run))

        hybrid_run = rrf_fuse_runs(
            [("JobBERT", jobbert_run), ("SPLADE", splade_run)],
            final_k=K_FINAL,
            rrf_k=FUSION_RRF_K,
        )
        runs.append(("JobBERT+SPLADE_RRF", hybrid_run))

        rerank_run = rerank_candidates_with_secondary_rrf(
            primary_candidates=jobbert_cand_run,
            secondary_full_run=splade_full_run,
            final_k=K_FINAL,
            rrf_k=FUSION_RRF_K,
            primary_weight=RERANK_JOBBERT_WEIGHT,
            secondary_weight=RERANK_SPLADE_WEIGHT,
        )
        runs.append(("JobBERT>>SPLADE_RRF", rerank_run))

    rows = []
    for name, run_df in runs:
        safe_name = name.lower().replace(">>", "_rerank_").replace("+", "_").replace(" ", "_")
        run_path = art_dir / f"run_dev_{lang}_{safe_name}_at{K_FINAL}.trec"
        save_trec_run(run_df, run_path, tag=f"{lang}_{safe_name}_at{K_FINAL}")
        metrics = evaluate_run(run_df, qrels, k=K_FINAL)
        row = {
            "lang": lang,
            "pipeline": name,
            "queries": len(topics),
            "docs": len(corpus),
            "run_path": str(run_path),
        }
        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows).sort_values(["lang", f"MAP@{K_FINAL}"], ascending=[True, False]).reset_index(drop=True)


def export_test_submission(lang: str, cfg: dict, pipeline_name: str, team_name: str, approach_id: str) -> Path:
    corpus, topics = load_task_a_inputs(cfg["test_dir"])
    topics_aug = ensure_augmented_topics(topics, lang, "test")
    art_dir = ART_ROOT / lang
    art_dir.mkdir(parents=True, exist_ok=True)
    topics_pt = build_query_view(topics_aug, "ORIGINAL")

    pipeline_name = pipeline_name.upper().replace(" ", "")
    jobbert_view_map = {
        "JOBBERT": "ORIGINAL",
        "JOBBERT_COMPACT_REWRITE": "COMPACT_REWRITE",
        "JOBBERT_PROFILE": "PROFILE",
        "JOBBERT_IDEAL_RESUME": "IDEAL_RESUME",
        "JOBBERT_PROFILE_IDEAL_RESUME": "PROFILE_IDEAL_RESUME",
        "JOBBERT_PROFILE_IDEAL_RESUME_SKILLS": "PROFILE_IDEAL_RESUME_SKILLS",
    }

    if pipeline_name == "BM25":
        pt_index = build_or_load_pt_index(corpus, art_dir / "pt_text_index_test")
        retriever = pt.terrier.Retriever(pt_index, wmodel="BM25")
        run_df = pt_run_to_standard_run(retriever.transform(topics_pt), len(corpus))
    elif pipeline_name in jobbert_view_map:
        run_df = build_jobbert_run_for_topics(
            corpus,
            build_query_view(topics_aug, jobbert_view_map[pipeline_name]),
            len(corpus),
        )
    elif pipeline_name in {"JOBBERT_COLBERT_RERANK", "JOBBERT>>COLBERT", "JOBBERT_COLBERT_COMPACT_RERANK", "JOBBERT+COMPACTREWRITE>>COLBERT"}:
        primary_full_run = build_jobbert_run_for_topics(corpus, topics_pt, len(corpus))
        primary_candidates = cut_run_to_k(primary_full_run, COLBERT_CAND_K)
        colbert_view = "COMPACT_REWRITE" if pipeline_name in {"JOBBERT_COLBERT_COMPACT_RERANK", "JOBBERT+COMPACTREWRITE>>COLBERT"} else COLBERT_QUERY_VIEW
        reranked_candidates = build_colbert_rerank_run(
            corpus=corpus,
            topics=build_query_view(topics_aug, colbert_view),
            primary_candidates=primary_candidates,
            final_k=COLBERT_CAND_K,
            model_name=COLBERT_MODEL_NAME,
            device=COLBERT_DEVICE,
            query_length=COLBERT_QUERY_LENGTH,
            document_length=COLBERT_DOCUMENT_LENGTH,
            query_batch_size=COLBERT_QUERY_BATCH_SIZE,
            document_batch_size=COLBERT_DOCUMENT_BATCH_SIZE,
        )
        run_df = splice_reranked_candidates(primary_full_run, reranked_candidates, final_k=len(corpus))
    elif pipeline_name in {"JOBBERT_COLBERT_CHUNKED_RERANK", "JOBBERT>>COLBERTCHUNKED"}:
        primary_full_run = build_jobbert_run_for_topics(corpus, topics_pt, len(corpus))
        primary_candidates = cut_run_to_k(primary_full_run, COLBERT_CAND_K)
        reranked_candidates = build_colbert_chunked_rerank_run(
            corpus=corpus,
            topics=build_query_view(topics_aug, COLBERT_QUERY_VIEW),
            primary_candidates=primary_candidates,
            final_k=COLBERT_CAND_K,
            model_name=COLBERT_MODEL_NAME,
            device=COLBERT_DEVICE,
            query_length=COLBERT_QUERY_LENGTH,
            document_length=COLBERT_DOCUMENT_LENGTH,
            query_batch_size=COLBERT_QUERY_BATCH_SIZE,
            document_batch_size=COLBERT_DOCUMENT_BATCH_SIZE,
            chunk_size_words=COLBERT_CHUNK_SIZE_WORDS,
            chunk_stride_words=COLBERT_CHUNK_STRIDE_WORDS,
            max_chunks_per_doc=COLBERT_MAX_CHUNKS_PER_DOC,
        )
        run_df = splice_reranked_candidates(primary_full_run, reranked_candidates, final_k=len(corpus))
    elif pipeline_name in {"JOBBERT_COMPACT_REWRITE_RRF", "JOBBERT+COMPACTREWRITE_RRF"}:
        compact_run = build_jobbert_run_for_topics(
            corpus,
            build_query_view(topics_aug, "COMPACT_REWRITE"),
            len(corpus),
        )
        original_run = build_jobbert_run_for_topics(corpus, topics_pt, len(corpus))
        run_df = rrf_fuse_runs(
            [("JobBERT", original_run), ("CompactRewrite", compact_run)],
            final_k=len(corpus),
            rrf_k=FUSION_RRF_K,
            weights={"JobBERT": COMPACT_RRF_JOBBERT_WEIGHT, "CompactRewrite": COMPACT_RRF_REWRITE_WEIGHT},
        )
    elif pipeline_name in {"JOBBERT_SPLADE_RRF", "JOBBERT+SPLADE_RRF", "SPLADE+JOBBERT_RRF"}:
        if not HAS_SPLADE:
            raise RuntimeError("Hybrid JobBERT+SPLADE was requested but pyt_splade is not installed.")
        jobbert_run = build_jobbert_run_for_topics(corpus, topics_pt, len(corpus))
        splade_index = build_or_load_splade_index(corpus, art_dir / "pt_splade_index_test")
        splade = pyt_splade.Splade()
        retriever = splade.query_encoder() >> pt.terrier.Retriever(splade_index, wmodel="Tf")
        splade_run = pt_run_to_standard_run(retriever.transform(topics_pt), len(corpus))
        run_df = rrf_fuse_runs(
            [("JobBERT", jobbert_run), ("SPLADE", splade_run)],
            final_k=len(corpus),
            rrf_k=FUSION_RRF_K,
        )
    elif pipeline_name in {"JOBBERT_SPLADE_RERANK", "JOBBERT>>SPLADE_RRF"}:
        if not HAS_SPLADE:
            raise RuntimeError("JobBERT>>SPLADE reranking was requested but pyt_splade is not installed.")
        jobbert_run = build_jobbert_run_for_topics(corpus, topics_pt, len(corpus))
        splade_index = build_or_load_splade_index(corpus, art_dir / "pt_splade_index_test")
        splade = pyt_splade.Splade()
        retriever = splade.query_encoder() >> pt.terrier.Retriever(splade_index, wmodel="Tf")
        splade_run = pt_run_to_standard_run(retriever.transform(topics_pt), len(corpus))
        run_df = rerank_candidates_with_secondary_rrf(
            primary_candidates=jobbert_run,
            secondary_full_run=splade_run,
            final_k=len(corpus),
            rrf_k=FUSION_RRF_K,
            primary_weight=RERANK_JOBBERT_WEIGHT,
            secondary_weight=RERANK_SPLADE_WEIGHT,
        )
    elif pipeline_name == "SPLADE":
        if not HAS_SPLADE:
            raise RuntimeError("SPLADE was requested but pyt_splade is not installed.")
        splade_index = build_or_load_splade_index(corpus, art_dir / "pt_splade_index_test")
        splade = pyt_splade.Splade()
        retriever = splade.query_encoder() >> pt.terrier.Retriever(splade_index, wmodel="Tf")
        run_df = pt_run_to_standard_run(retriever.transform(topics_pt), len(corpus))
    else:
        raise ValueError(f"Unsupported pipeline: {pipeline_name}")

    submission_run_path = SUBMISSION_ROOT / f"run_{cfg['submission_lang_pair']}_{sanitize_token(team_name)}.trec"
    save_trec_run(run_df, submission_run_path, tag=f"{sanitize_token(team_name)}_{sanitize_token(approach_id)}")
    return submission_run_path


In [ ]:
aug_preview = pd.concat(
    [
        summarize_augmentation(
            ensure_augmented_topics(
                load_task_a_inputs(cfg["development_dir"])[1],
                lang,
                "development",
            )
        ).assign(lang=lang)
        for lang, cfg in LANG_CONFIGS.items()
    ],
    ignore_index=True,
)
aug_preview[["lang", "qid", "title", "seniority", "skills", "compact_rewrite_text", "profile_text", "ideal_resume_text"]].head(6)


In [ ]:
dev_metrics = pd.concat(
    [run_dev_experiments(lang, cfg) for lang, cfg in LANG_CONFIGS.items()],
    ignore_index=True,
)
dev_metrics


In [ ]:
if RUN_TEST_SUBMISSION:
    SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)
    submission_paths = [
        export_test_submission(
            lang=lang,
            cfg=cfg,
            pipeline_name=TEST_PIPELINE,
            team_name=TEAM_NAME,
            approach_id=APPROACH_ID,
        )
        for lang, cfg in LANG_CONFIGS.items()
    ]
    zip_path = SUBMISSION_ROOT / f"{sanitize_token(TEAM_NAME)}_{sanitize_token(APPROACH_ID)}_{TEST_PIPELINE.lower()}_test_submission.zip"
    zip_submission(submission_paths, zip_path)
    print("Saved submission zip:", zip_path)
    print("Saved test runs:")
    for path in submission_paths:
        print(" ", path)
else:
    print("Set RUN_TEST_SUBMISSION = True to export test-set submission files.")
